Aqui vamos a probar un metodo que se llama "minirocket" que consiste en crear miles de redes convolucionales con distintos parametros aleatorios y comprobar cual da el mejor resultados. Son redes pequeñas y acaba siendo lineal, por lo q el tiempo de prediccion es pequeño, y puede llegar a ser muy preciso, por lo q es un candidato a darnos lo que buscamos.

Empezamos por impoar librerias e iputar los nans paa preparar los datos.

In [1]:
import numpy as np
import pandas as pd
import time
import warnings
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, classification_report
from sktime.transformations.panel.rocket import MiniRocketMultivariate
from competition_api import load_dataset

warnings.filterwarnings('ignore')

def imputar_nans(serie):
    df = pd.DataFrame(serie.T)
    df.interpolate(method='linear', limit_direction='both', inplace=True)
    return df.values.T

print("1. Cargando y limpiando datos (Train y Test Demo)...")
# Train
ds_train = load_dataset("data/train_100k.npz")
X_train_limpio = np.array([imputar_nans(m) for m in ds_train.series])
y_train = ds_train.labels

# Test
ds_test = load_dataset("data/test_demo.npz")
X_test_limpio = np.array([imputar_nans(m) for m in ds_test.series])
y_test = ds_test.labels

# sktime (MiniROCKET) exige que los datos estén en float64
X_train_mr = X_train_limpio.astype(np.float64)
X_test_mr = X_test_limpio.astype(np.float64)

print(f"Datos listos. Forma del Train: {X_train_mr.shape}")

1. Cargando y limpiando datos (Train y Test Demo)...
Datos listos. Forma del Train: (100000, 4, 256)


Entrenamos el miniRocket

In [2]:
print("2. Entrenando MiniROCKET (Generando ~10.000 filtros)...")
t0 = time.time()
minirocket = MiniRocketMultivariate(random_state=42)

# Ajustar y transformar el conjunto de entrenamiento
X_train_transformado = minirocket.fit_transform(X_train_mr)
t_rocket = time.time() - t0

print(f"-> Transformación completada en {t_rocket:.1f} segundos.")
print(f"-> Forma del Train tras MiniROCKET: {X_train_transformado.shape} (Fíjate en las ~10k variables)")

2. Entrenando MiniROCKET (Generando ~10.000 filtros)...
-> Transformación completada en 90.0 segundos.
-> Forma del Train tras MiniROCKET: (100000, 9996) (Fíjate en las ~10k variables)


Entrenamiento de la parte lineal

In [3]:
from sklearn.linear_model import RidgeClassifierCV
from sklearn.calibration import CalibratedClassifierCV
import time

print("3. Entrenando Ridge Classifier Balanceado y Calibrado...")
t0 = time.time()

# 1. Ridge con balanceo de clases para rescatar a la clase 6
ridge_base = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10), class_weight='balanced')

# 2. Calibración para obtener probabilidades reales (necesario para el ABSTAIN)
clasificador = CalibratedClassifierCV(estimator=ridge_base, cv=3)
clasificador.fit(X_train_transformado, y_train)

t_ridge = time.time() - t0
print(f"-> Entrenamiento del clasificador completado en {t_ridge:.1f} segundos.")

3. Entrenando Ridge Classifier Balanceado y Calibrado...
-> Entrenamiento del clasificador completado en 1172.4 segundos.


Test

In [6]:
print("4. EVALUACIÓN EN EL TEST DEMO...")

# Transformamos el test con los filtros ya generados
X_test_transformado = minirocket.transform(X_test_mr)

# Predecimos
preds_test = clasificador.predict(X_test_transformado)
acc_test = accuracy_score(y_test, preds_test)

# PRUEBA CRÍTICA DE TIEMPO (Simulando la API del concurso)
# Cogemos solo 1 muestra, manteniendo las dimensiones (1, 4, 256)
muestra_aislada = X_test_mr[0:1] 
t0_inf = time.perf_counter()
# En el concurso, el pipeline es: Transformar 1 muestra -> Predecir
muestra_trans = minirocket.transform(muestra_aislada)
_ = clasificador.predict(muestra_trans)
t_inf = time.perf_counter() - t0_inf

print("\n--- RESULTADOS DEFINITIVOS MINIROCKET ---")
print(f"Accuracy en TEST DEMO: {acc_test:.5f}")
print(f"Tiempo de inferencia (1 muestra): {t_inf:.6f} segundos (Límite: 0.5s)")

print("\n--- REPORTE POR CLASE ---")
print(classification_report(y_test, preds_test))

4. EVALUACIÓN EN EL TEST DEMO...

--- RESULTADOS DEFINITIVOS MINIROCKET ---
Accuracy en TEST DEMO: 0.66750
Tiempo de inferencia (1 muestra): 0.114647 segundos (Límite: 0.5s)

--- REPORTE POR CLASE ---
              precision    recall  f1-score   support

           0       0.61      0.80      0.69        50
           1       1.00      0.98      0.99        50
           2       0.94      0.58      0.72        50
           3       0.53      0.62      0.57        50
           4       0.84      0.82      0.83        50
           5       0.71      0.94      0.81        50
           6       0.23      0.06      0.10        50
           7       0.40      0.54      0.46        50

    accuracy                           0.67       400
   macro avg       0.66      0.67      0.64       400
weighted avg       0.66      0.67      0.64       400



Guardamos los pesos, requisito vital paa el concurso

In [7]:
import joblib
from pathlib import Path

print("Guardando los pesos del modelo para el concurso...")

# Guardamos los dos objetos entrenados en un solo archivo
ruta_pesos = Path.cwd() / "weights.pkl"
joblib.dump({"minirocket": minirocket, "ridge": clasificador}, ruta_pesos)

print(f"Archivo guardado en: {ruta_pesos}")

Guardando los pesos del modelo para el concurso...
Archivo guardado en: /home/miik02/uni/GDG-ML-Competition/GDG-ML-competition/weights.pkl
